# **M3. Cruce Semaforico Personalizado**

Emilio Alejandro González Huerta A01286440

Laurie Camila Hernández Pacheco A01286569

Hugo Edel Gamboa Sesma A00841605

Ángel Darío Marroquín Escobedo A01286669

Eugenio Moreno Vargas A01723720

---


## Descripción del cruce

El cruce modelado corresponde a la intersección real ubicada en torno al **Parque Primavera**
en Monterrey. La forma consiste en **Av. Ricardo Covarrubias** como eje
horizontal de doble sentido, que cruza con **Calle Independiente** (vertical, sentido norte)
en la parte central. En los extremos inferiores de Covarrubias se conectan dos ramas de
**Blvd. Primavera**: uno al suroeste (salida 94 VEH.) y otro al sureste (entrada 185 VEH.).

### Flujos modelados

| Origen | Dirección | Destino | Volumen Total |
|--------|-----------|---------|---------|
| Covarrubias izquierda | este, gira ↓ | Blvd. Primavera izq. | 94 VEH. |
| Covarrubias izquierda | recto este | Salida derecha Covarrubias | 407 VEH. |
| Covarrubias izquierda | este, gira ↑  | Independiente (norte) | 5 VEH. |
| Covarrubias derecha   | oeste, gira → hacia Independiente | Independiente (norte) | 5 VEH. |
| Covarrubias derecha   | recto oeste | Salida izq. Covarrubias | 478 VEH. |
| Covarrubias derecha   | oeste, gira → | Blvd. Primavera izq. | 94 VEH. |
| Blvd. Primavera derecha | norte, gira → | Salida este Covarrubias | 407 VEH. |
| Blvd. Primavera derecha      | norte, gira ← | Salida oeste Covarrubias | 478 VEH. |
| Blvd. Primavera derecha   | norte, gira ←, gira ↑ | Independiente (norte) | 5 VEH. |



## **Instalación**

In [ ]:
import agentpy as ap
import numpy as np
import matplotlib as mpl
mpl.rcParams["animation.embed_limit"]=1000
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from matplotlib.animation import FuncAnimation

from scipy.interpolate import splprep, splev
import IPython

# ─────────────────────────────────────────────────────────────────────────────
# PARÁMETROS GLOBALES
# ─────────────────────────────────────────────────────────────────────────────
SEED        = 6700
CAR_SPEED   = 5       # unidades por paso
COLLISION_D = 8       # distancia mínima entre autos

# Límites del mundo, escala 2:1
X_MIN, X_MAX = -175, 145
Y_MIN, Y_MAX =  -78,  78

## **Definición del Entorno**


El entorno de la simulación continua se representa mediante un **espacio vectorial continuo**
de 360 × 160 unidades, donde cada calle es una curva paramétrica suave generada con
**splines cúbicos** (scipy `splprep/splev`). A diferencia del modelo discreto en grid,
las posiciones de los vehículos son coordenadas flotantes `(x, y)` y las calles pueden
tener cualquier forma curvilínea.

El sistema de coordenadas usa ejes matemáticos estándar: `x` positivo hacia la derecha
e `y` positivo hacia arriba, con el origen aproximadamente en el centro de la intersección
principal.

### Calles definidas

Cada calle se define como una lista de **puntos de control** por los que scipy interpola
una curva suave con `k=3` (cúbica). La curva resultante se parametriza con `t ∈ [0, 1]`,
donde `t=0` es el inicio y `t=1` el fin del tramo.

| Calle | Puntos de control | Sentido |
|-------|-------------------|---------|
| Av. Covarrubias (carril inf) | `(-180,47)` → `(150,35)` | → este |
| Av. Covarrubias (carril sup) | `(-180,39)` → `(150,27)` | ← oeste |
| Calle Independiente | `(-36,7)` → `(-29,80)` | ↑ norte |
| Blvd. Primavera izquierda | `(-123,23)` → `(-100,-84)` | ↓ sur (salida 94 VEH.) |
| Blvd. Primavera derecha | `(78,0)` → `(106,-87)` | ↓ sur (entrada 185 VEH.) |

In [ ]:
WORLD_W      = 360
WORLD_H      = 160

def make_spline(control_points, n_samples=400):
    """
    Interpola una curva suave (spline cúbica) entre los puntos de control.
    Devuelve (xs, ys, tck) donde t ∈ [0, 1].
    """
    pts = np.array(control_points)
    k   = min(3, len(pts) - 1)
    tck, _ = splprep([pts[:, 0], pts[:, 1]], s=0, k=k)
    t_vals  = np.linspace(0, 1, n_samples)
    xs, ys  = splev(t_vals, tck)
    return xs, ys, tck

STREET_CONTROLS = {
    # Av. Covarrubias — carril superior (vehículos van de izq a der →)
    'Covarrubias_inf': [
        (-180, 47), (-140, 36), (-90, 22), (-35, 9),
        (0, 4), (40, 6), (80, 14), (150, 35)
    ],
    # Av. Covarrubias — carril inferior (vehículos van de der a izq ←)
    'Covarrubias_sup': [
        (-180, 39), (-140, 28), (-128, 25), (-90, 14), (-35, 1),
        (0, -4), (40, -2), (80, 6), (150, 27)
    ],
    # Calle Independiente — sube hacia el norte ↑
    'Independiente': [
        (-36, 7), (-34, 30), (-31, 60), (-29, 80)
    ],
    # Blvd. Primavera izquierda — baja hacia el sur ↓ (salida 94 VEH.)
    'Primavera_izq': [
        (-123, 23), (-123, 0), (-120, -20),
        (-116, -40), (-110, -60), (-100, -84)
    ],
    # Blvd. Primavera derecha — baja hacia el sur ↓ (entrada 185 VEH.)
    'Primavera_der': [
        (78, 0), (90, -20), (100, -40),
        (105, -60), (106, -87)
    ],
}

STREET_SPLINES = {}
for name, pts in STREET_CONTROLS.items():
    xs, ys, tck = make_spline(pts)
    STREET_SPLINES[name] = {'xs': xs, 'ys': ys, 'tck': tck}

# Longitud aproximada de cada spline (para normalizar velocidad)
STREET_LENGTHS = {}
for _name, _sp in STREET_SPLINES.items():
    STREET_LENGTHS[_name] = float(
        np.sum(np.hypot(np.diff(_sp['xs']), np.diff(_sp['ys'])))
    )

STREET_WIDTHS = {
    'Covarrubias_sup' : 3.5,
    'Covarrubias_inf' : 3.5,
    'Independiente'   : 4.2,
    'Primavera_izq'   : 4.2,
    'Primavera_der'   : 4.2,
}

## **Rutas de los Vehículos**

Por cada entrada posible de vehículos, existen 3 salidas distintas hacia otras calles.

In [ ]:
ROUTES = [
    # ── COVARRUBIAS IZQUIERDA → (morados) ────────────────────────────────────
    {
        # Entra por el extremo izquierdo de Cov_sup, gira sur en Primavera_izq
        'name'    : 'Cov_Izq_a_Primavera',
        'color'   : '#9b59b6',
        'segments': [
            {'spline': 'Covarrubias_sup', 't_start': 0.02,  't_end': 0.2},
            {'spline': 'Primavera_izq',   't_start': 0.02,  't_end': 1.0 },
        ]
    },
    {
        # Entra por el extremo izquierdo de Cov_sup, sale por el derecho
        'name'    : 'Cov_Izq_Recto',
        'color'   : '#6c3483',
        'segments': [
            {'spline': 'Covarrubias_sup', 't_start': 0.02, 't_end': 1.1},
        ]
    },
    {
        # Entra por el extremo izquierdo de Cov_sup, gira norte en Independiente
        'name'    : 'Cov_Izq_a_Independiente',
        'color'   : '#d2b4de',
        'segments': [
            {'spline': 'Covarrubias_sup', 't_start': 0.02,  't_end': 0.45},
            {'spline': 'Independiente',   't_start': 0.02,  't_end': 1.0 },
        ]
    },

    # ── COVARRUBIAS DERECHA ← (verdes) ───────────────────────────────────────
    {
        # Entra por el extremo derecho de Cov_inf, gira norte en Independiente
        'name'    : 'Cov_Der_a_Independiente',
        'color'   : '#27ae60',
        'segments': [
            {'spline': 'Covarrubias_inf', 't_start': 0.98,  't_end': 0.44},
            {'spline': 'Independiente',   't_start': 0.05,  't_end': 1.0 },
        ]
    },
    {
        # Entra por el extremo derecho de Cov_inf, sale por el izquierdo
        'name'    : 'Cov_Der_Recto',
        'color'   : '#1a5c38',
        'segments': [
            {'spline': 'Covarrubias_inf', 't_start': 0.98, 't_end': 0.0},
        ]
    },
    {
        # Entra por el extremo derecho de Cov_inf, gira sur en Primavera_izq
        'name'    : 'Cov_Der_a_Primavera',
        'color'   : '#a9dfbf',
        'segments': [
            {'spline': 'Covarrubias_inf', 't_start': 0.98,  't_end': 0.17},
            {'spline': 'Primavera_izq',   't_start': 0.0,  't_end': 1.0 },
        ]
    },

    # ── BLVD. PRIMAVERA DERECHA ↑ (amarillos) ────────────────────────────────
    {
        # Sube por Primavera_der, gira derecha (este) en Cov_sup
        'name'    : 'Primavera_Der_a_Der',
        'color'   : '#f1c40f',
        'segments': [
            {'spline': 'Primavera_der',   't_start': 0.9,  't_end': 0.0 },
            {'spline': 'Covarrubias_sup', 't_start': 0.77, 't_end': 1.0 },
        ]
    },
    {
        # Sube por Primavera_der, gira izquierda (oeste) en Cov_inf
        'name'    : 'Primavera_Der_a_Izq',
        'color'   : '#d4ac0d',
        'segments': [
            {'spline': 'Primavera_der',   't_start': 0.9,  't_end': 0.0 },
            {'spline': 'Covarrubias_inf', 't_start': 0.77, 't_end': 0.0 },
        ]
    },
    {
        # Sube por Primavera_der, gira oeste en Cov_inf, gira norte en Independiente
        'name'    : 'Primavera_Der_a_Independiente',
        'color'   : '#f9e79f',
        'segments': [
            {'spline': 'Primavera_der',   't_start': 0.9,  't_end': 0.0 },
            {'spline': 'Covarrubias_inf', 't_start': 0.77, 't_end': 0.43},
            {'spline': 'Independiente',   't_start': 0.0,  't_end': 1.0 },
        ]
    },
]

# Grupos por punto de entrada
COV_IZQ_ROUTES   = [r for r in ROUTES if r['name'].startswith('Cov_Izq')]
COV_DER_ROUTES   = [r for r in ROUTES if r['name'].startswith('Cov_Der')]
PRIMAVERA_ROUTES = [r for r in ROUTES if r['name'].startswith('Primavera')]
ROUTE_GROUPS     = [
    ('Cov_Izq',   COV_IZQ_ROUTES),
    ('Cov_Der',   COV_DER_ROUTES),
    ('Primavera', PRIMAVERA_ROUTES),
]

#pesos dia
ROUTE_WEIGHTS = {
    'Cov_Izq_a_Primavera'           : 0.155,
    'Cov_Izq_Recto'                 : 0.837,
    'Cov_Izq_a_Independiente'       : 0.009,
    'Cov_Der_a_Independiente'       : 0.004,
    'Cov_Der_Recto'                 : 0.906,
    'Cov_Der_a_Primavera'           : 0.090,
    'Primavera_Der_a_Der'           : 0.649,
    'Primavera_Der_a_Izq'           : 0.351,
    'Primavera_Der_a_Independiente' : 0.000,
}

"""
# pesos tarde:
ROUTE_WEIGHTS = {
    'Cov_Izq_a_Primavera'           : 0.176,
    'Cov_Izq_Recto'                 : 0.822,
    'Cov_Izq_a_Independiente'       : 0.003,
    'Cov_Der_a_Independiente'       : 0.000,
    'Cov_Der_Recto'                 : 0.831,
    'Cov_Der_a_Primavera'           : 0.090,
    'Primavera_Der_a_Der'           : 0.588,
    'Primavera_Der_a_Izq'           : 0.412,
    'Primavera_Der_a_Independiente' : 0.000,
}
"""
"""
# pesos noche:
ROUTE_WEIGHTS = {
    'Cov_Izq_a_Primavera'           : 0.155,
    'Cov_Izq_Recto'                 : 0.839,
    'Cov_Izq_a_Independiente'       : 0.006,
    'Cov_Der_a_Independiente'       : 0.049,
    'Cov_Der_Recto'                 : 0.865,
    'Cov_Der_a_Primavera'           : 0.090,
    'Primavera_Der_a_Der'           : 0.653,
    'Primavera_Der_a_Izq'           : 0.339,
    'Primavera_Der_a_Independiente' : 0.008,
}
"""


## **Definición del Agente Vehículo**

Cada vehículo es un agente de la clase `Car` implementada con **agentpy**. A diferencia
del modelo discreto, los vehículos no ocupan celdas sino que existen en el espacio continuo
con posición flotante y se desplazan a lo largo de curvas paramétricas.

### Atributos del agente

| Atributo | Tipo | Descripción |
|----------|------|-------------|
| `x, y` | `float` | Posición actual en el espacio continuo |
| `seg_idx` | `int` | Índice del segmento de ruta activo |
| `t` | `float` | Parámetro de posición en el spline actual `[0, 1]` |
| `t_end` | `float` | Valor de `t` donde termina el segmento actual |
| `t_dir` | `±1.0` | Dirección de recorrido del spline (`+1` avanza, `-1` retrocede) |
| `segments` | `list` | Secuencia de segmentos de spline que conforman la ruta |
| `state` | `str` | Estado del agente: `moving` o `done` |
| `dx, dy` | `float` | Vector de dirección normalizado (usado para marcador y colisiones) |

### Movimiento

El vehículo avanza incrementando el parámetro `t` en cada paso mediante `_advance_t`.
La velocidad se normaliza por la **longitud real** del spline activo, garantizando que
todos los vehículos se desplacen a la misma velocidad física sin importar la longitud
de su calle. Al llegar al `t_end` del segmento activo, el agente carga el siguiente
segmento de su ruta con `_load_segment`, o se marca como `done` si no hay más segmentos.
El agente también se marca como `done` si su posición sale de los límites del mundo
`(X_MIN, X_MAX, Y_MIN, Y_MAX)`.

### Reglas de interacción

El agente aplica dos reglas por paso:

**No colisión** — antes de avanzar, revisa todos los autos activos en el mismo segmento
de spline. Si hay uno a distancia menor a `COLLISION_D` en la dirección de marcha
(verificado con producto punto), el agente espera sin moverse.

**Zona de merge** — cuando el agente está a menos de `MERGE_WAIT_D` unidades del final
de su segmento actual (a punto de incorporarse a otra calle), verifica que no haya
ningún vehículo cercano al punto de empalme en ninguna dirección dentro de un radio
`check_distance`. Si el acceso no está despejado, el agente cede el paso y espera.

In [ ]:
# Distancia de la zona de espera antes de entrar a Covarrubias
MERGE_WAIT_D = 6   # unidades — ajusta según el mapa

def is_merge_zone(car):
    """
    Devuelve True si el auto está cerca del final de su segmento actual
    (a punto de hacer un empalme a otra calle).
    """
    seg = car.segments[car.seg_idx]
    remaining = abs(car.t_end - car.t) * STREET_LENGTHS[seg['spline']]
    return remaining < MERGE_WAIT_D

MERGE_PRIORITY = {
    'Primavera_Der_a_Independiente' : 1,
    'Primavera_Der_a_Der'           : 1,
    'Primavera_Der_a_Izq'           : 1,
    'Cov_Izq_a_Independiente'       : 2,
    'Cov_Der_a_Independiente'       : 2,
    'Cov_Izq_a_Primavera'           : 3,
    'Cov_Der_a_Primavera'           : 3,
    'Cov_Izq_Recto'                 : 3,
    'Cov_Der_Recto'                 : 3,
}

# Rutas que deben esperar más atrás antes de entrar a Independiente
# para no llegar a bloquear al amarillo claro en el merge
EARLY_WAIT_ROUTES = {'Cov_Izq_a_Independiente', 'Cov_Der_a_Independiente'}
EARLY_WAIT_DISTANCE = 22.0   # distancia ampliada de revisión para estas rutas

def merge_is_clear(car, others, check_distance=12.0):
    next_idx = car.seg_idx + 1
    if next_idx >= len(car.segments):
        return True

    next_spline = car.segments[next_idx]['spline']
    next_t      = car.segments[next_idx]['t_start']
    next_t_end  = car.segments[next_idx]['t_end']
    tck         = STREET_SPLINES[next_spline]['tck']
    mx, my      = eval_spline_at_t(tck, next_t)

    t_dir    = 1.0 if next_t_end >= next_t else -1.0
    ddx, ddy = spline_direction_at_t(tck, next_t, t_dir)

    my_priority = MERGE_PRIORITY.get(car.route, 99)

    # Rutas que van a Independiente desde Covarrubias esperan más atrás
    effective_check = EARLY_WAIT_DISTANCE if car.route in EARLY_WAIT_ROUTES else check_distance

    for other in others:
        if other is car or other.state == 'done':
            continue
        dist = np.hypot(other.x - mx, other.y - my)
        if dist > effective_check:
            continue
        dot = (other.x - mx) * ddx + (other.y - my) * ddy
        if dot <= 0:
            continue
        other_priority = MERGE_PRIORITY.get(other.route, 99)
        if other_priority <= my_priority:
            return False

    return True

    next_spline = car.segments[next_idx]['spline']
    next_t      = car.segments[next_idx]['t_start']
    next_t_end  = car.segments[next_idx]['t_end']
    tck         = STREET_SPLINES[next_spline]['tck']
    mx, my      = eval_spline_at_t(tck, next_t)

    t_dir    = 1.0 if next_t_end >= next_t else -1.0
    ddx, ddy = spline_direction_at_t(tck, next_t, t_dir)

    for other in others:
        if other is car or other.state == 'done':
            continue

        dist = np.hypot(other.x - mx, other.y - my)
        if dist > check_distance:
            continue

        # Bloquear si el otro está adelante en cualquier dirección cercana
        dot = (other.x - mx) * ddx + (other.y - my) * ddy
        if dot > 0:
            return False

    return True

def eval_spline_at_t(tck, t):
    t = float(np.clip(t, 0.0, 1.0))
    x, y = splev(t, tck)
    return float(x), float(y)


def spline_direction_at_t(tck, t, t_dir=1.0):
    t  = float(np.clip(t, 0.0, 1.0))
    dt = 0.005 * t_dir
    if t + dt > 1.0:
        dt = -0.005
    if t + dt < 0.0:
        dt = 0.005
    x0, y0 = splev(t,      tck)
    x1, y1 = splev(t + dt, tck)
    dx, dy  = float(x1 - x0), float(y1 - y0)
    norm    = np.hypot(dx, dy) + 1e-9
    return dx / norm, dy / norm


class Car(ap.Agent):
    """
    Agente vehículo en espacio continuo con splines.

    Atributos:
      x, y       — posición actual
      seg_idx    — índice del segmento activo
      t          — parámetro en el spline activo [0, 1]
      t_end      — t objetivo del segmento
      t_dir      — +1 o -1 según la dirección de recorrido
      segments   — lista de segmentos de la ruta
      state      — 'moving' | 'done'
      route      — nombre de la ruta
      color      — color de visualización
      dx, dy     — vector de dirección actual (normalizado)
    """

    def setup(self):
        self.x        = 0.0
        self.y        = 0.0
        self.seg_idx  = 0
        self.t        = 0.0
        self.t_end    = 1.0
        self.t_dir    = 1.0
        self.segments = []
        self.state    = 'moving'
        self.route    = ''
        self.color    = 'white'
        self.dx       = 1.0
        self.dy       = 0.0

    def _load_segment(self, idx):
        seg         = self.segments[idx]
        self.t      = seg['t_start']
        self.t_end  = seg['t_end']
        self.t_dir  = 1.0 if self.t_end >= self.t else -1.0
        tck         = STREET_SPLINES[seg['spline']]['tck']

        # NO actualizar x,y aquí — dejar que _advance_t lo haga gradualmente
        # Solo actualizar dirección
        self.dx, self.dy = spline_direction_at_t(tck, self.t, self.t_dir)

        # Snap de posición solo si el salto es pequeño
        new_x, new_y = eval_spline_at_t(tck, self.t)
        dist = np.hypot(new_x - self.x, new_y - self.y)
        if dist < COLLISION_D * 3:
            self.x, self.y = new_x, new_y
        # Si el salto es grande, mantener posición actual un paso más

    def _advance_t(self, speed):
        """
        Mueve el auto a lo largo del spline actual.
        La velocidad se normaliza por la longitud real de la curva.
        """
        seg      = self.segments[self.seg_idx]
        tck      = STREET_SPLINES[seg['spline']]['tck']
        length   = STREET_LENGTHS[seg['spline']]
        dt       = (speed / (length + 1e-9)) * self.t_dir
        new_t    = self.t + dt

        if self.t_dir > 0 and new_t >= self.t_end:
            new_t       = self.t_end
            reached_end = True
        elif self.t_dir < 0 and new_t <= self.t_end:
            new_t       = self.t_end
            reached_end = True
        else:
            reached_end = False

        self.t           = float(np.clip(new_t, 0.0, 1.0))
        self.x, self.y   = eval_spline_at_t(tck, self.t)
        self.dx, self.dy = spline_direction_at_t(tck, self.t, self.t_dir)
        return reached_end
    
    def step(self, others, speed):
        if self.state == 'done':
            return
        
        if self.x < X_MIN or self.x > X_MAX or self.y < Y_MIN or self.y > Y_MAX:
            self.state = 'done'
            return

        # Colisión normal (misma calle)
        for other in others:
            if other is self or other.state == 'done':
                continue
            my_spline    = self.segments[self.seg_idx]['spline']
            other_spline = other.segments[other.seg_idx]['spline']
            if my_spline != other_spline:
                continue
            dist = np.hypot(self.x - other.x, self.y - other.y)
            if dist < COLLISION_D:
                dot = (other.x - self.x) * self.dx + (other.y - self.y) * self.dy
                if dot > 0:
                    return

        # Zona de espera antes de empalme: ceder paso
        if is_merge_zone(self) and not merge_is_clear(self, others):
            return   # esperar hasta que Covarrubias esté despejada

        reached_end = self._advance_t(speed)

        if reached_end:
            next_idx = self.seg_idx + 1
            if next_idx < len(self.segments):
                self.seg_idx = next_idx
                self._load_segment(next_idx)
            else:
                self.state = 'done'

## **Agentes Semáforo y Sensor**

Se añade un **semáforo adaptivo 3-way** en la intersección de Blvd. Primavera con Av. Covarrubias.

### Fases del semáforo

| Fase | Covarrubias | Primavera | Duración |
|------|-------------|-----------|----------|
| `green_cov` | 🟢 Verde | 🔴 Rojo | indefinido, hasta que haya demanda |
| `yellow` | 🟡 Amarillo | 🟡 Amarillo | 5 pasos (transición) |
| `green_pri` | 🔴 Rojo | 🟢 Verde | 20 pasos, o hasta que no haya demanda |

### Lógica de demanda

Cada auto de Primavera registra su demanda al llegar a la zona de stop.
El semáforo cambia a verde para Primavera **solo si hay demanda activa** y ya
pasó el mínimo de verde Covarrubias. Al terminar el verde de Primavera,
si no hay más demanda, se queda en `green_cov` indefinidamente sin ciclar en vacío.

### Deadlock Independiente

Los autos de Covarrubias que van a Independiente (`Cov_Izq_a_Independiente`,
`Cov_Der_a_Independiente`) esperan con un radio ampliado (`EARLY_WAIT_DISTANCE = 22`)
antes del merge, para no llegar a bloquear al amarillo claro que tiene prioridad 1.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SEMÁFORO ADAPTIVO 3-WAY  —  Blvd. Primavera / Av. Covarrubias
# ─────────────────────────────────────────────────────────────────────────────
GREEN_COV_MIN   = 50   # pasos mínimos de verde Covarrubias antes de ceder
GREEN_PRI_STEPS = 30   # pasos máximos de verde Primavera (si sigue habiendo demanda, renueva)
YELLOW_STEPS    = 10    # pasos de transición amarillo

# ── MODO DE SEMÁFORO ─────────────────────────────────────────────────────────────
# Activa UNO. El adaptivo usa demanda de sensores; el fijo cicla por tiempo.

FIXED_CYCLE = None   # adaptivo (comportamiento normal)
"""
FIXED_CYCLE = 60     # semáforo fijo: alterna cada 60 pasos sin importar demanda
"""


# Posición visual de los semáforos (ajustables)
TL_POS_COV_E = (88,  30)
TL_POS_COV_W = (60,  18)
TL_POS_PRI   = (82, -52)

# ── Zona de stop Primavera_der ────────────────────────────────────────────────
_join_tck     = STREET_SPLINES['Primavera_der']['tck']
_jx, _jy      = eval_spline_at_t(_join_tck, 0.05)
STOP_LINE_POS = (float(_jx), float(_jy))
STOP_LINE_R   = 18

# ── Zonas de stop Covarrubias (una por carril) ────────────────────────────────
_cov_inf_tck = STREET_SPLINES['Covarrubias_inf']['tck']
_cix, _ciy   = eval_spline_at_t(_cov_inf_tck, 0.77)
COV_STOP_INF = (float(_cix), float(_ciy))

_cov_sup_tck = STREET_SPLINES['Covarrubias_sup']['tck']
_csx, _csy   = eval_spline_at_t(_cov_sup_tck, 0.77)
COV_STOP_SUP = (float(_csx), float(_csy))

COV_STOP_R = 16

# Registro global de demandas activas de Primavera
# Cada auto de Primavera agrega su id cuando llega a stop, lo quita cuando cruza
_primavera_demands = set()


def _car_near_stop_line(car):
    """¿El auto de Primavera está antes de la línea de pare?"""
    seg = car.segments[car.seg_idx]
    if seg['spline'] != 'Primavera_der':
        return False
    dx = STOP_LINE_POS[0] - car.x
    dy = STOP_LINE_POS[1] - car.y
    if np.hypot(dx, dy) > STOP_LINE_R:
        return False
    dot = dx * car.dx + dy * car.dy
    return dot > 0


def _covarrubias_near_junction(car):
    """¿El auto de Covarrubias está antes del punto de cruce con Primavera?"""
    seg    = car.segments[car.seg_idx]
    spline = seg['spline']
    if spline == 'Covarrubias_inf':
        stop = COV_STOP_INF
    elif spline == 'Covarrubias_sup':
        stop = COV_STOP_SUP
    else:
        return False
    dx = stop[0] - car.x
    dy = stop[1] - car.y
    if np.hypot(dx, dy) > COV_STOP_R:
        return False
    dot = dx * car.dx + dy * car.dy
    return dot > 0


class TrafficLight(ap.Agent):
    """Semáforo adaptivo 3-way basado en demanda real."""

    def setup(self):
        self.phase         = 'green_cov'
        self.phase_timer   = 0
        self.phase_history = []

    def has_demand(self):
        return len(_primavera_demands) > 0

    def tick(self):
        self.phase_timer += 1

        if FIXED_CYCLE is not None:
            # ── Semáforo fijo: ignora demanda, cicla por tiempo ──────────────
            period = FIXED_CYCLE * 2 + YELLOW_STEPS * 2
            pos    = self.phase_timer % period
            if pos < FIXED_CYCLE:
                self.phase = 'green_cov'
            elif pos < FIXED_CYCLE + YELLOW_STEPS:
                self.phase = 'yellow'
            elif pos < FIXED_CYCLE * 2 + YELLOW_STEPS:
                self.phase = 'green_pri'
            else:
                self.phase = 'yellow_to_cov'

        else:
            # ── Semáforo adaptivo (original) ─────────────────────────────────
            if self.phase == 'green_cov':
                # Solo cambia si hay demanda activa Y se cumplió el mínimo
                if self.has_demand() and self.phase_timer >= GREEN_COV_MIN:
                    self.phase       = 'yellow'
                    self.phase_timer = 0

            elif self.phase == 'yellow':
                if self.phase_timer >= YELLOW_STEPS:
                    self.phase       = 'green_pri'
                    self.phase_timer = 0

            elif self.phase == 'green_pri':
                # Termina cuando se acaba el tiempo O ya no hay demanda
                if self.phase_timer >= GREEN_PRI_STEPS or not self.has_demand():
                    self.phase       = 'yellow_to_cov'
                    self.phase_timer = 0

            elif self.phase == 'yellow_to_cov':
                if self.phase_timer >= YELLOW_STEPS:
                    self.phase       = 'green_cov'
                    self.phase_timer = 0

        self.phase_history.append(self.phase)


class Sensor(ap.Agent):
    """
    Sensor de demanda: cada auto de Primavera se registra al llegar
    a la zona de stop y se desregistra al cruzar.
    """

    def setup(self):
        self.traffic_light = None

    def update_demands(self, cars):
        global _primavera_demands
        active_ids = set()

        for c in cars:
            if not c.route.startswith('Primavera') or c.state != 'moving':
                continue
            # El auto está esperando en la zona de stop
            if _car_near_stop_line(c):
                _primavera_demands.add(id(c))
                active_ids.add(id(c))
            # Si ya cruzó la línea de stop (dot <= 0 o fuera del radio),
            # ya no cuenta como demanda
            else:
                _primavera_demands.discard(id(c))

        # Limpiar ids de autos que ya salieron de la simulación
        _primavera_demands &= {id(c) for c in cars if c.state == 'moving'}


## **Modelo Multiagente**

El modelo `IntersectionModel` hereda de `ap.Model` y coordina el entorno, la generación
de vehículos y la actualización del sistema en cada paso de tiempo.

### Inicialización

Al arrancar, el modelo crea un generador de números aleatorios con semilla fija (`seed`)
para garantizar reproducibilidad, una lista vacía de agentes `Car`, y los contadores de
métricas `total_created` y `total_completed`. También inicializa el historial de snapshots
que se usará para la animación.

### Generación de vehículos (`spawn_cars`)

En cada paso, el modelo intenta generar un vehículo por cada grupo de entrada
(`Cov_Izq`, `Cov_Der`, `Primavera`) de forma independiente. La probabilidad de aparición
es configurable por grupo mediante `group_prob`. Si el grupo es seleccionado, se elige
una ruta específica dentro del grupo usando los pesos de `ROUTE_WEIGHTS`, que reflejan
la distribución real de giros observada en el aforo. El vehículo no se genera si hay
otro auto a menos de `COLLISION_D` unidades del punto de entrada.

### Actualización por paso (`step`)

Cada paso sigue cuatro etapas en orden:

1. **Spawn** — intentar generar nuevos vehículos en los puntos de entrada.
2. **Movimiento** — actualizar la posición de cada vehículo activo en orden aleatorio,
   evitando que la posición en la lista otorgue ventaja sistemática a algún agente.
3. **Retiro** — eliminar los vehículos cuyo estado es `done` y contabilizarlos en
   `total_completed`.
4. **Snapshot** — guardar el estado `(x, y, color, dx, dy)` de todos los vehículos
   activos para su posterior visualización y animación.

### Métricas finales

Al terminar la simulación, el modelo reporta el número de vehículos creados, los que
completaron su recorrido y el **throughput**: porcentaje de vehículos que lograron
cruzar el área del modelo respecto al total generado.

In [ ]:
class IntersectionModel(ap.Model):

    def setup(self):
        self.rng             = np.random.default_rng(self.p.seed)
        self.cars            = ap.AgentList(self, 0, Car)
        self.total_created   = 0
        self.total_completed = 0
        self.history         = []

        # Limpiar demandas globales al iniciar
        _primavera_demands.clear()

        self.traffic_light        = TrafficLight(self)
        self.sensor               = Sensor(self)
        self.sensor.traffic_light = self.traffic_light

    def spawn_cars(self):
        for group_name, group in ROUTE_GROUPS:
            prob = self.p.group_prob.get(group_name)
            if self.rng.random() >= prob:
                continue

            weights = np.array([ROUTE_WEIGHTS[r['name']] for r in group], dtype=float)
            weights /= weights.sum()
            idx   = int(self.rng.choice(len(group), p=weights))
            route = group[idx]

            car          = Car(self)
            car.segments = route['segments']
            car.state    = 'moving'
            car.route    = route['name']
            car.color    = route['color']
            car.seg_idx  = 0
            car._load_segment(0)

            too_close = any(
                np.hypot(car.x - o.x, car.y - o.y) < COLLISION_D
                for o in self.cars if o.state == 'moving'
            )
            if too_close:
                continue

            self.cars.append(car)
            self.total_created += 1

    def step(self):
        self.spawn_cars()

        # Sensor actualiza demandas, luego semáforo decide fase
        self.sensor.update_demands(list(self.cars))
        self.traffic_light.tick()
        phase = self.traffic_light.phase

        speed     = self.p.get('car_speed', CAR_SPEED)
        indices   = self.rng.permutation(len(self.cars))
        cars_list = list(self.cars)

        for i in indices:
            if i >= len(cars_list):
                continue
            c = cars_list[i]

            # Primavera se detiene cuando Covarrubias tiene verde o amarillo
            if c.route.startswith('Primavera') and phase in ('green_cov', 'yellow'):
                if _car_near_stop_line(c):
                    continue

            # Covarrubias se detiene cuando Primavera tiene verde o amarillo
            if not c.route.startswith('Primavera') and phase in ('green_pri', 'yellow'):
                if _covarrubias_near_junction(c):
                    continue

            c.step(cars_list, speed)

        done = [c for c in self.cars if c.state == 'done']
        self.total_completed += len(done)
        self.cars = ap.AgentList(self, [c for c in self.cars if c.state == 'moving'])

        self.history.append(
            [(c.x, c.y, c.color, c.dx, c.dy) for c in self.cars]
            + [('__tl__', phase)]
        )

    def end(self):
        print('\n=== Resultados de la Simulación ===')
        print(f'  Pasos simulados           : {self.p.steps}')
        print(f'  Vehículos creados         : {self.total_created}')
        print(f'  Vehículos que cruzaron    : {self.total_completed}')
        if self.total_created > 0:
            print(f'  Throughput                : {self.total_completed/self.total_created*100:.1f}%')


## **Ejecución**

In [ ]:

# mañna:
PARAMETERS = {
    'steps'      : 500,   
    'seed'       : SEED,
    'car_speed'  : CAR_SPEED,
    'group_prob' : {
        'Cov_Izq'   : 0.057,
        'Cov_Der'   : 0.076,
        'Primavera' : 0.031,
    }
}

"""
# tarde:
PARAMETERS = {
    'steps'      : 500,
    'seed'       : SEED,
    'car_speed'  : CAR_SPEED,
    'group_prob' : {
        'Cov_Izq'   : 0.064,
        'Cov_Der'   : 0.074,
        'Primavera' : 0.074,
    }
}
"""
"""

# noche:
PARAMETERS = {
    'steps'      : 500,
    'seed'       : SEED,
    'car_speed'  : CAR_SPEED,
    'group_prob' : {
        'Cov_Izq'   : 0.081,
        'Cov_Der'   : 0.074,
        'Primavera' : 0.041,
    }
}
"""


model   = IntersectionModel(PARAMETERS)
results = model.run()
history = model.history
print(f'Pasos en historial: {len(history)}')

## **Simulación**

In [ ]:
TL_COLORS = {
    'green_cov'    : {'cov': '#00e676', 'pri': '#f44336'},
    'yellow'       : {'cov': '#ffee58', 'pri': '#ffee58'},
    'yellow_to_cov': {'cov': '#ffee58', 'pri': '#ffee58'},
    'green_pri'    : {'cov': '#f44336', 'pri': '#00e676'},
}

PARQUE_VERTICES = [
    (-120, 30), (-40,  0), (-30,  0), (70,   0),
    (90,  -20), (100, -40), (110, -60), (110, -80),
    (-100, -80), (-120, -30), (-120, 30),
]


def draw_background(ax):
    ax.set_facecolor('#ebebeb')
    parque = Polygon(PARQUE_VERTICES, facecolor='#c2f0d0',
                     edgecolor='#5a8a6a', linewidth=0, zorder=0)
    ax.add_patch(parque)
    ax.text(-15, -40, 'PARQUE\nPrimavera', color='#77b59c',
            fontsize=14, ha='center', va='center', fontweight='bold', zorder=1)


def draw_streets(ax):
    for name, spline in STREET_SPLINES.items():
        w = STREET_WIDTHS.get(name, 3)
        ax.plot(spline['xs'], spline['ys'],
                color='#898f96', lw=w * 6.25,
                solid_capstyle='round', zorder=2)
        ax.plot(spline['xs'], spline['ys'],
                color='#a9b8c8', lw=w * 6,
                solid_capstyle='round', zorder=3)
    for name in ['Covarrubias_sup', 'Covarrubias_inf']:
        s = STREET_SPLINES[name]
        ax.plot(s['xs'], s['ys'], color='#e8e0c0',
                lw=0.8, ls='--', alpha=0.8, zorder=4)


def draw_labels(ax):
    kw = dict(fontweight='bold', alpha=0.35, color='#3d4043', zorder=5)
    ax.text( 80,  10, 'Av. Ricardo Covarrubias', fontsize=10,
             ha='center', va='center', rotation=15,  **kw)
    ax.text(-120, 25, 'Av. Ricardo Covarrubias', fontsize=10,
             ha='center', va='center', rotation=-15, **kw)
    ax.text(-33,  45, 'Independiente', fontsize=8,
             ha='center', va='center', rotation=80,  **kw)
    ax.text(-115, -55, 'Blvd.\nPrimavera', fontsize=7,
             ha='center', va='center', rotation=-80, **kw)
    ax.text( 100, -55, 'Blvd.\nPrimavera', fontsize=7,
             ha='center', va='center', rotation=80,  **kw)


def draw_traffic_lights(ax, phase):
    colors = TL_COLORS.get(phase, TL_COLORS['green_cov'])
    for pos, label in [(TL_POS_COV_E, 'COV E'), (TL_POS_COV_W, 'COV W')]:
        ax.scatter(*pos, s=150, color=colors['cov'], edgecolors='black',
                   linewidths=1.2, marker='s', zorder=8)
        ax.text(pos[0], pos[1] + 6, label, fontsize=5,
                ha='center', color='#333', zorder=9)
    ax.scatter(*TL_POS_PRI, s=150, color=colors['pri'], edgecolors='black',
               linewidths=1.2, marker='s', zorder=8)
    ax.text(TL_POS_PRI[0], TL_POS_PRI[1] - 7, 'PRI', fontsize=5,
            ha='center', color='#333', zorder=9)


def direction_marker(dx, dy):
    if abs(dx) > abs(dy):
        return '>' if dx > 0 else '<'
    return 'v' if dy < 0 else '^'


def draw_frame(ax, snapshot, step_num):
    ax.clear()
    draw_background(ax)
    draw_streets(ax)
    draw_labels(ax)

    tl_phase = 'green_cov'
    vehicles  = []
    for item in snapshot:
        if isinstance(item, tuple) and len(item) == 2 and item[0] == '__tl__':
            tl_phase = item[1]
        else:
            vehicles.append(item)

    draw_traffic_lights(ax, tl_phase)

    phase_labels = {
        'green_cov'    : 'Covarrubias 🟢  |  Primavera 🔴',
        'yellow'       : 'Transición  🟡  |  Transición 🟡',
        'yellow_to_cov': 'Transición  🟡  |  Transición 🟡',
        'green_pri'    : 'Covarrubias 🔴  |  Primavera 🟢',
    }
    ax.text(X_MIN + 5, Y_MIN + 5, phase_labels.get(tl_phase, ''),
            fontsize=7, color='#333', va='bottom', zorder=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    for (x, y, color, dx, dy) in vehicles:
        marker = direction_marker(dx, dy)
        ax.scatter(x, y, s=70, marker=marker, color=color,
                   edgecolors='black', linewidths=0.6, zorder=6)

    patches = [
        mpatches.Patch(color='#9b59b6', label='Cov. Izq →'),
        mpatches.Patch(color='#27ae60', label='Cov. Der ←'),
        mpatches.Patch(color='#f1c40f', label='Primavera ↑'),
    ]
    ax.legend(handles=patches, loc='upper right', fontsize=7,
              framealpha=0.85, facecolor='#f5f5f5')

    ax.set_title(f'Cruce Parque Primavera — Paso {step_num}  |  Activos: {len(vehicles)}',
                 fontsize=9)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_aspect('equal')
    ax.tick_params(colors='gray')


In [ ]:
#Activar para animacion
#Advertencia, se tarda bastante ~ 30s por cada 250 pasos
"""
fig_anim, ax_anim = plt.subplots(figsize=(10, 6))

def animate(frame):
    draw_frame(ax_anim, history[frame], frame)

anim = FuncAnimation(fig_anim, animate, frames=len(history), repeat=True)
plt.close()
IPython.display.HTML(anim.to_jshtml(fps=15))
"""


## **Métricas : Throughput**

In [ ]:
gp = PARAMETERS['group_prob']

print('=' * 50)
print('         MÉTRICAS DE LA SIMULACIÓN')
print('=' * 50)
print(f'  Semilla RNG              : {PARAMETERS["seed"]}')
print(f'  Pasos simulados          : {PARAMETERS["steps"]}')
print()
print('  Probabilidad de spawn por grupo:')
for group, prob in gp.items():
    print(f'    {group:12s} {prob:.0%}')
print()
print('  Pesos por ruta:')
for name, w in ROUTE_WEIGHTS.items():
    print(f'    {name:35s} {w:.0%}')
print('-' * 50)
print(f'  Vehículos CREADOS        : {model.total_created}')
print(f'  Vehículos que CRUZARON   : {model.total_completed}')
print(f'  Vehículos aún en tránsito: {len(model.cars)}')
if model.total_created > 0:
    pct = model.total_completed / model.total_created * 100
    print(f'  Throughput               : {pct:.1f}%')
print('=' * 50)

# Gráfica de vehículos activos por paso
active_per_step = [len(snap) for snap in history]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(active_per_step, color='deepskyblue', lw=2)
ax.fill_between(range(len(active_per_step)), active_per_step,
                alpha=0.2, color='deepskyblue')
ax.set_title('Vehículos activos en el cruce por paso de tiempo', fontsize=12)
ax.set_xlabel('Paso')
ax.set_ylabel('Número de vehículos')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f8f8')
plt.tight_layout()
plt.show()